# Decison Trees

Here the classification is done through a tree-like structure by splitting data points into separate classes using logical yes/no questions based on conditions on feature values or categories.
This allows to explore and asess different pathways in the data.
Now  for choosing the best features to split the data points can be done using the following matrics:
### Gini impurity:
It measures the likelihood of a randomly chosen data point being incorrectly classified if it were randomly labeled according to the distribution of classes in the node.

This is the default critetion for data splitting in this model
 

The gini index:

$ Gini(t)=1-\sum{[p(j|t)]^2}$ 

The total impurity:

$Gini_{split}=\sum{\frac{n_i}{n}Gini(t)}$


$t$ : The given node

$p(j|t)$ : The probability (proportion) of class $j$ present in node $t$

$n_i$ : Number of samples in child node $i$

$n$ : Total number of samples in the parent node

A Gini score of 0 means the node is completely "pure" (all samples of the current data subset belong to the exact same class).
The model tries to achieve the minimuum gini score possible on the data split to get a split that generates a pure leaf node


### Entropy
This tell us the uncertainity in the dataset at a specific node
$$H = -\sum_{k=1}^{K} p_k \log p_k$$

If all of the data points of the data subset belong to only one class at this node then no uncertainity is there and when the data points are equally distributed between the two classes at this node then the uncertainity is at the maximum (1)

### Info gain
This calculates how much the entropy decreases if we go ahead with a specific data split

$IG = H(parent) - \sum_{j} \frac{|S_j|}{|S|} H(S_j)$

We use this to get better dataset splitting conditions.

Higher is better



## The problem of overfitting

If we dont put guardrails on the splitting process this model has a definite chance of overfitting. to preevent that , we can set some control paramters:

#### The max depth
Specifies how far below the root node can the tree split, stops the splitting once it reaaches the max_depth

#### min_samples_split
This tells the model the minumum number of data samples it should have before going ahead with the split.

#### min_sample_leaf
ensures each of the leaf node(nodes at the end of the tree having giving us the class decision) has a specififc number/ fraction of data samples , if the leaf node data samples are below that number then that split is disregarded.
This prevents model from creatiing extremely specific rules that cause overfit

#### max_features

The number of features to consider when looking for the best split


## Post-Pruning 

Instead of stopping the tree early (pre-pruning), we can build a fully grown tree, and then remove the parts that are not useful

This approach allows the model to initially capture all possible patterns and later simplify itself by removing unnecessary complexity.


### Cost Complexity Pruning

This is the most common method used for post-pruning.

We define a cost function that balances how well the tree fits the data and the tree's complexity 

$$
\text{Cost} = \text{Error} + \alpha \times \text{Tree Size}
$$

- Error → measures how well the tree fits the data (e.g., MSE in regression, misclassification in classification)  

- Tree Size → number of leaf nodes   

- $\alpha$  → regularization parameter that controls how much we penalize complexity  



### Workflow

1. Start with the fully grown tree  
2. Evaluate the cost of the tree  
3. Try removing a branch (subtree)  
4. Replace it with a leaf node (usually using the mean or majority value)  
5. Recalculate the cost  

- If the cost decreases, we keep the pruned version  
- If not, we keep the original subtree  

This process is repeated until no further improvement is possible


### Some issues
The DT is very sensitive to the training data 
Overfit on high dimensional data
### Docs:

https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = load_iris()
X = data.data   
y = data.target 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt_classifier = DecisionTreeClassifier(max_depth=2, random_state=42)

dt_classifier.fit(X_train, y_train)

y_pred = dt_classifier.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")


Model Accuracy: 96.67%


# Regression Tree

Here, the model builds a tree-like structure by splitting the dataset into smaller subsets using yes/no questions on feature values, with the goal of **grouping similar target values together**.

This allows the model to explore different regions of the data and assign a **numerical prediction** to each region.

---

At each node, the model checks if the data split results in groups having more similar values.

So we measure how close the target values are within each node


## Prediction in a leaf node

Each leaf node stores a single value, which is usually the **mean (average)** of all target values in that node.

So, when a new input reaches a leaf the prediction is the average of training values in that region


## Splitting criteria 

To decide the best split, we use **error-based metrics** instead of Gini or Entropy.



### Mean Squared Error (MSE)

This is the most commonly used criterion.

$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y})^2
$$

- $y_i$ = actual value  
- $\hat{y}$ = predicted value (mean of node)  


if values in a node are **very similar** → MSE is low  and if values are **spread out** → MSE is high  



### Split MSE

$$
MSE_{split} = \sum \frac{n_i}{n} \cdot MSE_i
$$

- $n_i$ = number of samples in child node  
- $n$ = number of samples in parent node  
- $MSE_i$ = error in each child node  

The model chooses the split that gives the **lowest total MSE**


## Alternative criteria

### Mean Absolute Error (MAE)

$$
MAE = \frac{1}{n} \sum |y_i - \hat{y}|
$$

- less sensitive to outliers  
- more robust in noisy data  


### Variance Reduction 

Another way to understand splitting is through **variance**.

$$
Var = \frac{1}{n} \sum (y_i - \bar{y})^2
$$

So a good split should reduce the variance in the child nodes compared to the parent, resultine in tighter grouping


$$
\text{Variance Reduction} = Var(parent) - \sum \frac{n_i}{n} Var(child_i)
$$




In [4]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

data = fetch_california_housing()
X = data.data  
y = data.target 


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


dt = DecisionTreeRegressor(max_depth=5, random_state=42)

dt.fit(X_train, y_train)


y_pred = dt.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2) Score: {r2:.2f}")


C:\Users\ALIENWARE\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\datasets\_base.py:1519: UserWarning: Retry downloading from url: https://ndownloader.figshare.com/files/5976036
  warnings.warn(f"Retry downloading from url: {remote.url}")


Mean Squared Error (MSE): 0.52
R-squared (R2) Score: 0.60
